# 🇦🇺 Australian Aircraft Register - Dataset Processing Pipeline

## Dataset Information

**Dataset Name**: Australian Aircraft Register (ACRFTREG)
**Source**: Civil Aviation Safety Authority (CASA) (https://www.casa.gov.au/)
**Original Data Source**: CASA
**Copyright**: Commonwealth of Australia

### Context
This dataset provides a registry of aircraft in Australia, including details about registrations, types, owners, and other related information.

### Content
- **Contains**: Records of registered aircraft in Australia.
- **Fields**: Includes various columns such as registration, aircraft type, owner details, etc.

### Data Source
- Directly from CASA: https://services.casa.gov.au/CSV/acrftreg.csv

### Original Citation
Civil Aviation Safety Authority (CASA) - https://www.casa.gov.au/

---

## 1. Import Required Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import json
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# ML libraries
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split

print("✅ Libraries imported successfully!")
print(f"Pandas version: {pd.__version__}")
print(f"NumPy version: {np.__version__}")

## 2. Load Raw Data

In [ ]:
from pathlib import Path
import pandas as pd # Ensure pandas is imported

# Define dataset paths
dataset_path = Path('.')
raw_path = dataset_path / 'raw'
processed_path = dataset_path / 'processed'
features_path = dataset_path / 'features'

# Create directories if they don't exist
raw_path.mkdir(exist_ok=True)
processed_path.mkdir(exist_ok=True)
features_path.mkdir(exist_ok=True)

# Load the raw data
# Prefer loading from the dataset's local raw/ folder
try:
    raw_file = raw_path / 'acrftreg.csv'
    alt_file = dataset_path / 'raw' / 'acrftreg.csv'
    if raw_file.exists():
        df_raw = pd.read_csv(raw_file)
        print(f"✅ Loaded raw data from '{raw_file}' successfully!")
    elif alt_file.exists():
        df_raw = pd.read_csv(alt_file)
        print(f"✅ Loaded raw data from '{alt_file}' successfully!")
    else:
        raise FileNotFoundError(f"'{raw_file}' and '{alt_file}' not found.")
except FileNotFoundError as e:
    print(f"❌ Error: {e}")
    df_raw = pd.DataFrame()  # Create an empty DataFrame to avoid errors later

# Display dataset info
if not df_raw.empty:
    print(f"\n📊 Dataset Shape: {df_raw.shape}")
    print(f"Rows: {df_raw.shape[0]}, Columns: {df_raw.shape[1]}")
    print(f"\n📋 First few rows:")
    print(df_raw.head())
    print(f"\n📋 Data types:")
    print(df_raw.dtypes)
    print(f"\n📋 Missing values:")
    print(df_raw.isnull().sum())

✅ Loaded raw data from '/content/acrftreg.csv' successfully!

📊 Dataset Shape: (16600, 44)
Rows: 16600, Columns: 44

📋 First few rows:
  Mark                         Manu  Type     Model Serial    MTOW  engnum  \
0  22A    CIRRUS DESIGN CORPORATION   NaN      SR22  10637  1633.0       1   
1  22V       ROBINSON HELICOPTER CO   NaN  R22 BETA   3766   622.0       1   
2  4QP  BELL TEXTRON CANADA LIMITED   NaN       429  57536  3402.0       2   
3  5QP  BELL TEXTRON CANADA LIMITED   NaN       429  57537  3402.0       2   
4  6QP  BELL TEXTRON CANADA LIMITED   NaN       429  57480  3402.0       2   

                      Engmanu     Engtype   Engmodel  ...  \
0     CONTINENTAL MOTORS INC.      Piston   IO-550-N  ...   
1            TEXTRON LYCOMING      Piston  O-360-J2A  ...   
2  PRATT & WHITNEY CANADA INC  Turboshaft    PW207D1  ...   
3  PRATT & WHITNEY CANADA INC  Turboshaft    PW207D1  ...   
4  PRATT & WHITNEY CANADA INC  Turboshaft    PW207D1  ...   

                             

/tmp/ipykernel_5436/119822646.py:18: DtypeWarning: Columns (17,25,40,41) have mixed types. Specify dtype option on import or set low_memory=False.
  df_raw = pd.read_csv('/content/acrftreg.csv')


## 3. Exploratory Data Analysis (EDA)

In [16]:
# Statistical summary
print("📊 STATISTICAL SUMMARY")
print("=" * 80)
print(df_raw.describe())

# Null values analysis
print("\n\n🔍 MISSING VALUES ANALYSIS")
print("=" * 80)
missing_stats = pd.DataFrame({
    'Column': df_raw.columns,
    'Missing_Count': df_raw.isnull().sum(),
    'Missing_Percentage': (df_raw.isnull().sum() / len(df_raw) * 100).round(2)
})
print(missing_stats[missing_stats['Missing_Count'] > 0].sort_values('Missing_Percentage', ascending=False))

# Display data types and non-null counts
print("\n\n💡 DATA TYPES AND NON-NULL COUNTS")
print("=" * 80)
df_raw.info()

📊 STATISTICAL SUMMARY
       Type           MTOW        engnum  gear  CoAcatc      Yearmanu  \
count   0.0   16600.000000  16600.000000   0.0      0.0  16600.000000   
mean    NaN    5050.236507      1.089819   NaN      NaN   1988.722590   
std     NaN   22757.825613      0.538481   NaN      NaN     20.939497   
min     NaN       0.000000      0.000000   NaN      NaN   1925.000000   
25%     NaN     800.000000      1.000000   NaN      NaN   1974.000000   
50%     NaN    1156.000000      1.000000   NaN      NaN   1986.000000   
75%     NaN    1814.000000      1.000000   NaN      NaN   2007.000000   
max     NaN  572000.000000     16.000000   NaN      NaN   2026.000000   

       Regexpirydate  
count            0.0  
mean             NaN  
std              NaN  
min              NaN  
25%              NaN  
50%              NaN  
75%              NaN  
max              NaN  


🔍 MISSING VALUES ANALYSIS
                                        Column  Missing_Count  \
Type                

## 4. Data Cleaning & Preprocessing

In [27]:
import numpy as np # Added this import

# Create a copy for processing
df_processed = df_raw.copy()

print("🧹 STARTING DATA CLEANING PROCESS")
print("=" * 80)

# Step 1: Remove duplicates
duplicates_before = df_processed.shape[0]
df_processed = df_processed.drop_duplicates()
duplicates_removed = duplicates_before - df_processed.shape[0]
print(f"✓ Step 1: Removed {duplicates_removed} duplicate rows")

# Step 2: Standardize column names
df_processed.columns = df_processed.columns.str.strip().str.lower().str.replace(' ', '_').str.replace('[^a-z0-9_]', '', regex=True)
print(f"✓ Step 2: Standardized column names")

# Step 3: Handle missing values
print(f"\n✓ Step 3: Handling missing values")

# Numeric columns - fill with median
numeric_cols = df_processed.select_dtypes(include=[np.number]).columns
for col in numeric_cols:
    if df_processed[col].isnull().sum() > 0:
        median_val = df_processed[col].median()
        df_processed[col] = df_processed[col].fillna(median_val)
        print(f"  - Filled '{col}' with median: {median_val:.2f}")

# Categorical columns - fill with mode
categorical_cols = df_processed.select_dtypes(include=['object']).columns
for col in categorical_cols:
    if df_processed[col].isnull().sum() > 0:
        mode_val = df_processed[col].mode()[0] if not df_processed[col].mode().empty else 'Unknown'
        df_processed[col] = df_processed[col].fillna(mode_val)
        print(f"  - Filled '{col}' with mode: {mode_val}")

print(f"\n✅ DATA CLEANING COMPLETE!")
print(f"Final shape: {df_processed.shape}")
print(f"Final missing values: {df_processed.isnull().sum().sum()}")

🧹 STARTING DATA CLEANING PROCESS
✓ Step 1: Removed 0 duplicate rows
✓ Step 2: Standardized column names

✓ Step 3: Handling missing values
  - Filled 'type' with median: nan
  - Filled 'gear' with median: nan
  - Filled 'coacatc' with median: nan
  - Filled 'regexpirydate' with median: nan
  - Filled 'engmanu' with mode: TEXTRON LYCOMING
  - Filled 'regholdadd1' with mode: 10 Bourke Rd
  - Filled 'regholdadd2' with mode: 10 Darcy St
  - Filled 'regholdsuburb' with mode: MASCOT
  - Filled 'regholdstate' with mode: NSW
  - Filled 'regholdpostcode' with mode: 2020
  - Filled 'regholdcountry' with mode: Australia
  - Filled 'regopadd1' with mode: 10 Bourke Rd
  - Filled 'regopadd2' with mode: Kittyhawk Lane
  - Filled 'regopsuburb' with mode: MASCOT
  - Filled 'regopstate' with mode: NSW
  - Filled 'regoppostcode' with mode: 3194
  - Filled 'regopcountry' with mode: Australia
  - Filled 'coacata' with mode: Active (Normal)
  - Filled 'coacatb' with mode: Active (Experimental)
  - Filled 't

## 5. Feature Engineering

Feature engineering is skipped for this dataset as no specific predictive task was identified. If a specific task (e.g., predicting aircraft age, popularity) were defined, custom features could be engineered here.

## 6. Machine Learning Preparation

Machine learning preparation steps (like train/test split, feature scaling, encoding for models) are skipped as this dataset is being processed for general cleaning and exploration, not a specific ML task at this stage. If an ML task is defined, this section would be populated accordingly.

## 7. Save Processed Datasets

In [23]:
from datetime import datetime
import json # Ensure json is imported

print("💾 SAVING PROCESSED DATASETS")
print("=" * 80)

# Save cleaned data
df_processed.to_csv(processed_path / 'aircraft_register_clean.csv', index=False)
print(f"✓ Saved: {processed_path / 'aircraft_register_clean.csv'}")

# Create metadata file
metadata = {
    "dataset_name": "Australian Aircraft Register (ACRFTREG)",
    "source": "Civil Aviation Safety Authority (CASA)",
    "original_source": "CASA (https://www.casa.gov.au/)",
    "copyright": "Commonwealth of Australia",
    "description": "Registry of aircraft in Australia, including details about registrations, types, owners.",
    "created_date": str(datetime.now()),
    "data_cleaning": {
        "duplicates_removed": int(duplicates_removed),
        "missing_values_handled": "Median for numeric, Mode for categorical",
        "outliers_removed_method": "None (skipped)",
        "categorical_encoding": "None (skipped)"
    },
    "features_engineered": "None (skipped)",
    "total_samples": len(df_processed),
    "total_features": df_processed.shape[1],
    "files": {
        "raw": "/content/acrftreg.csv",
        "cleaned": "processed/aircraft_register_clean.csv",
        "metadata": "metadata.json"
    }
}

# Save metadata
with open(dataset_path / 'metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)
print(f"✓ Saved: {dataset_path / 'metadata.json'}")

print(f"\n✅ ALL DATASETS SAVED SUCCESSFULLY!")
print(f"\n📁 Dataset Structure:")
print(f"   raw/")
print(f"   ├── acrftreg.csv (original data)")
print(f"   processed/")
print(f"   ├── aircraft_register_clean.csv (cleaned)")
print(f"   metadata.json (dataset documentation)")

💾 SAVING PROCESSED DATASETS
✓ Saved: processed/aircraft_register_clean.csv
✓ Saved: metadata.json

✅ ALL DATASETS SAVED SUCCESSFULLY!

📁 Dataset Structure:
   raw/
   ├── acrftreg.csv (original data)
   processed/
   ├── aircraft_register_clean.csv (cleaned)
   metadata.json (dataset documentation)


## 8. Quick Start - Using the Processed Data

### Load the processed data:
```python
import pandas as pd

# Load processed data
df_processed = pd.read_csv('processed/aircraft_register_clean.csv')
print("Loaded processed aircraft register data.")
print(df_processed.head())
```

### Metadata:
Load `metadata.json` to understand the dataset structure and preprocessing steps.


## Processed Australian Aircraft Register Dataset - Summary

### Overview
This notebook demonstrates a pipeline for processing the Australian Aircraft Register dataset from the Civil Aviation Safety Authority (CASA), providing a cleaned and structured version ready for analysis.

### Dataset Details
**Source**: Civil Aviation Safety Authority (CASA) - https://www.casa.gov.au/
**Original Data**: https://services.casa.gov.au/CSV/acrftreg.csv

### Files Included
Your current directory now contains:
```
. (root directory)
├── raw/
│   └── acrftreg.csv                    (Original raw data)
├── processed/
│   └── aircraft_register_clean.csv     (Cleaned and preprocessed data)
└── metadata.json                       (Documentation of the dataset and processing steps)
```

### Data Processing Highlights
- **Standardization**: Column names have been standardized to lowercase, snake_case format for consistency.
- **Duplicate Removal**: 0 duplicate rows were found and removed, indicating the raw data had unique entries.
- **Missing Value Handling**:
    - **Numeric Columns**: Missing values in numeric columns were imputed with their respective medians. For columns where all values were missing (e.g., 'type', 'gear', 'coacatc', 'regexpirydate'), they remain `NaN` as their median is also `NaN`.
    - **Categorical Columns**: Missing values in categorical columns were imputed with their most frequent value (mode). If a mode was not found (e.g., all values missing), 'Unknown' was used.
- **Final State**: The dataset has a final shape of (16600, 44) with 66400 remaining missing values, primarily in columns that were entirely empty in the raw data.
- **Skipped Steps**: Feature Engineering and Machine Learning Preparation were skipped as no specific predictive task was defined for this general data processing pipeline.

### Usage
The `aircraft_register_clean.csv` file is now ready for direct use in various analytical tasks. You can load it using pandas:
```python
import pandas as pd

df_aircraft = pd.read_csv('processed/aircraft_register_clean.csv')
print(df_aircraft.head())
```

This processed dataset is suitable for:
- **Exploratory Data Analysis (EDA)**: Gaining insights into aircraft types, manufacturers, ownership, and registration patterns.
- **Reporting**: Generating statistical summaries or custom reports on the Australian aircraft fleet.
- **Data Integration**: Combining with other aviation-related datasets for richer, more complex analyses.
- **Machine Learning Preparation**: Serving as a clean base for future machine learning tasks, should a specific problem be identified.
